In [1]:
import pandas as pd
import numpy as np
import json
import warnings

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "../data/raw/UNSW_NB15_training-set.csv"
)

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

df.head()

Dataset loaded successfully!
Original Shape: (175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [3]:
missing_values = df.isnull().sum()

print("Total Missing Values:", missing_values.sum())

display(missing_values[missing_values > 0])

Total Missing Values: 0


Series([], dtype: int64)

In [4]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

print("Infinite Values:",
      np.isinf(df[numeric_columns]).sum().sum())

df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Infinite values converted to NaN.")

Infinite Values: 0
Infinite values converted to NaN.


In [5]:
print("Exact Duplicates:", df.duplicated().sum())

print("Duplicates Without ID:",
      df.drop(columns=["id"]).duplicated().sum())

Exact Duplicates: 0
Duplicates Without ID: 67601


In [6]:
df_clean = df.drop(columns=["id", "attack_cat"])

print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)

df_clean.head()

Original Shape: (175341, 45)
Cleaned Shape: (175341, 43)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087490,252,...,1,1,1,0,0,0,1,1,0,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,62,...,1,1,2,0,0,0,1,6,0,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,62,...,1,1,3,0,0,0,2,6,0,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,62,...,1,1,3,1,1,0,2,1,0,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373826,254,...,2,1,40,0,0,0,2,39,0,0


In [7]:
X = df_clean.drop(columns=["label"])

y = df_clean["label"]

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Classes:", y.unique())

Input Features Shape: (175341, 42)
Target Shape: (175341,)

Target Classes: [0 1]


In [8]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

Numerical Features: 39
['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Categorical Features: 3
['proto', 'service', 'state']


In [9]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline created.")

Numerical pipeline created.


In [10]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

print("Categorical pipeline created.")

Categorical pipeline created.


In [11]:
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Scaled preprocessor created.")

Scaled preprocessor created.


In [12]:
tree_numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", tree_numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Tree-based preprocessor created.")

Tree-based preprocessor created.


In [13]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

feature_info = {
    "target": "label",
    "total_features": X.shape[1],
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "feature_names": X.columns.tolist(),
    "class_labels": {
        "0": "Normal Traffic",
        "1": "Malicious Traffic"
    }
}

with open(PROCESSED_DIR / "feature_info.json", "w") as file:
    json.dump(feature_info, file, indent=4)

print("Feature information saved successfully!")

Feature information saved successfully!


In [14]:
df_clean.to_csv(
    PROCESSED_DIR / "cleaned_training_data.csv",
    index=False
)

print("Cleaned training dataset saved!")
print("Final Shape:", df_clean.shape)

Cleaned training dataset saved!
Final Shape: (175341, 43)
